In [1]:
! pip install markdown-it-py mdit_plain pypdf
! pip install gdown

In [2]:
from haystack import Pipeline
from haystack.components.routers import FileTypeRouter
from haystack.components.converters import PyPDFToDocument,MarkdownToDocument,TextFileToDocument
from haystack.components.joiners import DocumentJoiner
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.document_stores.in_memory import InMemoryDocumentStore
from haystack.components.preprocessors import DocumentSplitter, DocumentCleaner
from haystack.components.writers import DocumentWriter


In [3]:
import gdown

url = "https://drive.google.com/drive/folders/1n9yqq5Gl_HWfND5bTlrCwAOycMDt5EMj"
output_dir = "recipe_files"

gdown.download_folder(url, quiet=True, output=output_dir)

['recipe_files\\vegan_flan_recipe.md',
 'recipe_files\\vegan_keto_eggplant_recipe_fixed.pdf',
 'recipe_files\\vegan_sunflower_hemp_cheese_recipe.txt']

In [4]:
document_store = InMemoryDocumentStore()
file_router = FileTypeRouter(mime_types = ["text/plain" , "application/pdf" , "text/markdown"])
pdf_converter = PyPDFToDocument()
text_converter = TextFileToDocument()
markdown_converter = MarkdownToDocument()
document_joiner = DocumentJoiner()

In [5]:
document_splitter = DocumentSplitter(split_by="word",split_length=150, split_overlap=50)
document_cleaner = DocumentCleaner()

In [6]:
document_embedder = SentenceTransformersDocumentEmbedder(model = "sentence-transformers/all-MiniLM-L6-v2")
document_writer = DocumentWriter(document_store=document_store)

In [7]:
indexing_pipeline = Pipeline()

indexing_pipeline.add_component("file_router" , file_router)
indexing_pipeline.add_component("pdf_converter" , pdf_converter)
indexing_pipeline.add_component("text_converter" , text_converter)
indexing_pipeline.add_component("markdown_converter" , markdown_converter)
indexing_pipeline.add_component("document_joiner" , document_joiner)
indexing_pipeline.add_component("document_splitter" , document_splitter)
indexing_pipeline.add_component("document_cleaner" , document_cleaner)
indexing_pipeline.add_component("document_embedder" , document_embedder)
indexing_pipeline.add_component("document_writer" , document_writer)


ValueError: Component named Document_joiner not found in the pipeline.

In [8]:
indexing_pipeline.connect("file_router.text/plain","text_converter")
indexing_pipeline.connect("file_router.application/pdf","pdf_converter")
indexing_pipeline.connect("file_router.text/markdown","markdown_converter")
indexing_pipeline.connect("pdf_converter","document_joiner")
indexing_pipeline.connect("text_converter","document_joiner")
indexing_pipeline.connect("markdown_converter","document_joiner")
indexing_pipeline.connect("document_joiner","document_cleaner")
indexing_pipeline.connect("document_cleaner","document_splitter")
indexing_pipeline.connect("document_splitter","document_embedder")
indexing_pipeline.connect("document_embedder","document_writer")

🚅 Components
  - file_router: FileTypeRouter
  - pdf_converter: PyPDFToDocument
  - text_converter: TextFileToDocument
  - markdown_converter: MarkdownToDocument
  - document_joiner: DocumentJoiner
  - document_splitter: DocumentSplitter
  - document_cleaner: DocumentCleaner
  - document_embedder: SentenceTransformersDocumentEmbedder
  - document_writer: DocumentWriter
🛤️ Connections
  - file_router.text/plain -> text_converter.sources (List[Union[str, Path, ByteStream]])
  - file_router.application/pdf -> pdf_converter.sources (List[Union[str, Path, ByteStream]])
  - file_router.text/markdown -> markdown_converter.sources (List[Union[str, Path, ByteStream]])
  - pdf_converter.documents -> document_joiner.documents (List[Document])
  - text_converter.documents -> document_joiner.documents (List[Document])
  - markdown_converter.documents -> document_joiner.documents (List[Document])
  - document_joiner.documents -> document_cleaner.documents (List[Document])
  - document_splitter.docum

In [ ]:
indexing_pipeline.run({"file_router":{list(Path(output_dir).glob("**/*"))}})